In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("customer_shopping_behavior.csv")
df.head(3)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [6]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [7]:
snake_case = df.copy()
def convert_snake_case(df:pd.DataFrame) -> pd.DataFrame:
  df.columns = [col.lower().replace(" ", "_") for col in df.columns]
  return df
snake_case = convert_snake_case(snake_case)
snake_case.head(3)

,customer_id,age,gender,item_purchased,category,purchase_amount_(usd),location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly


In [8]:
snake_case["review_rating"] = snake_case.groupby("category")["review_rating"].transform(lambda x: x.fillna(x.median()))

In [9]:
snake_case.isnull().sum()

,0
customer_id,0
age,0
gender,0
item_purchased,0
category,0
purchase_amount_(usd),0
location,0
size,0
color,0
season,0


In [10]:
snake_case = snake_case.rename(columns={"purchase_amount_(usd)": "purchase_amount"})

In [11]:
snake_case.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [12]:
labels = ["Young adult", "Adult", "Middle-Aged", "Senior"]
snake_case["age_group"] = pd.qcut(snake_case["age"], q=4, labels=labels)

In [13]:
snake_case[["age", "age_group"]].head(3)

,age,age_group
0,55,Middle-Aged
1,19,Young adult
2,50,Middle-Aged


In [14]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

snake_case["purchase_frequency_days"] = snake_case["frequency_of_purchases"].map(frequency_mapping)
snake_case[["frequency_of_purchases", "purchase_frequency_days"]].head(5)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365


In [15]:
(snake_case["discount_applied"] == snake_case["promo_code_used"]).all()

np.True_

In [16]:
snake_case = snake_case.drop(columns=["promo_code_used"])

In [17]:
import sqlite3

In [18]:
pip install sqlalchemy

In [19]:
from sqlalchemy import create_engine
engine = create_engine("sqlite:///customer_behavior.db")
engine.connect()
table_name = "customer"
snake_case.to_sql(table_name, engine, if_exists="replace", index=False)
print(f"Successfully loaded table {table_name} into db customer_behavior.db")

Successfully loaded table customer into db customer_behavior.db
